# V127 - Corrected Top1 action selector, dual frozen encoders (nested rebuild)

Prioritas 1 dari audit `results/leader_audit/FINDINGS_AND_NEXT_MODEL.md`. Semua pemodelan
delegasi ke `nb_code/v127_pipeline.py` yang diuji di `tests/test_v127_pipeline.py`:
rantai anchor champion-structured dibangun ulang per outer fold tanpa label validasi,
label aksi = gain swap eksak, fitur koordinat, konsensus dua head, maksimal satu swap.
Deviasi terdokumentasi di modul (tanpa intent bundle / mesin B-state; evaluasi OOF semua
user; proteksi test = mask V25). Gate gagal berarti byte-copy champion. Tanpa submission.


In [1]:
import hashlib
import json
import os
import sys
import time
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = bool(int(os.getenv("V127_SMOKE", "0")))
SEED = 6161
E5_ID = "intfloat/multilingual-e5-base"
NOMIC_ID = "nomic-ai/nomic-embed-text-v2-moe"
EXP_FILE = "443785fb534df9820ce57bfe450a2768881625bf7ae37aa849dd0d9f2afc957d"

ROOT = Path("/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender")
for cand in (ROOT / "nb_code", Path("nb_code"), Path("../nb_code")):
    if (cand / "v127_pipeline.py").exists():
        sys.path.insert(0, str(cand))
        break
import v127_pipeline as vp

cfg = vp.Config(seed=SEED, smoke=SMOKE,
                outer_splits=2 if SMOKE else 5, inner_splits=2 if SMOKE else 3,
                calibration_trees=20 if SMOKE else 150, ranker_trees=20 if SMOKE else 150,
                word_features=5000, char_features=8000, threads=4,
                bootstraps=200 if SMOKE else 2000)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data"), Path("../data"), ROOT / "data"]
DATA = next((c for c in CANDIDATES if (c / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CH = next((c for c in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                       Path("results/v42/submission_v42_primary.csv"),
                       ROOT / "results/v42/submission_v42_primary.csv"] if c.exists()), None)
assert CH is not None, "champion file not found"
assert sha256(CH.read_bytes()).hexdigest() == EXP_FILE, "champion file changed"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ROOT / "results/v127")
OUT.mkdir(parents=True, exist_ok=True)
T = {}
print(f"SMOKE={SMOKE} DATA={DATA} OUT={OUT}")


SMOKE=False DATA=../data OUT=/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v127


## Load, audit inputs, provenance caches

CSV organizer dimuat via modul (grup prefix, strata, assessment, docs). Cache embedding
diverifikasi bentuk dan pipeline (re-encode probe, cosine > 0.999) sebelum dipakai.


In [2]:
t0 = time.time()
data = vp.load_data(DATA)
n_train, n_test = data.n_train, data.n_test
docs = data.docs
ch = pd.read_csv(CH)
assert ch["user_id"].tolist() == data.user_ids[n_train:]
print(f"groups={len(set(data.groups))} train={n_train} test={n_test}")

import torch
from sentence_transformers import SentenceTransformer
E5C = ROOT / "results/v58/e5_users.npy"
NOC = ROOT / "results/v124/nomic_users.npy"
assert E5C.exists() and NOC.exists(), "shared embedding caches missing"
E5U = np.asarray(np.load(E5C), dtype=np.float32)
NOU = np.asarray(np.load(NOC), dtype=np.float32)
assert E5U.shape == (n_train + n_test, 768) and NOU.shape == (n_train + n_test, 768)
probe_idx = [0, 7, n_train - 1, n_train, n_train + n_test - 1]
e5t = SentenceTransformer(E5_ID)
pe = np.asarray(e5t.encode(["query: " + docs[i] for i in probe_idx], normalize_embeddings=True))
ce = (pe * E5U[probe_idx]).sum(1)
print(f"e5 probe cosine: {np.round(ce, 6)}")
assert (ce > 0.999).all(), "e5 cache pipeline mismatch"
del e5t
torch.cuda.empty_cache()
from transformers import AutoModel, AutoTokenizer
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
pn = []
with torch.no_grad():
    for i in probe_idx:
        e = nTok("search_query: " + docs[i], padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        pn.append((v / v.norm()).cpu().numpy().ravel())
pn = np.array(pn)
cn = (pn * (NOU[probe_idx] / (np.linalg.norm(NOU[probe_idx], axis=1, keepdims=True) + 1e-12))).sum(1)
print(f"nomic probe cosine: {np.round(cn, 6)}")
assert (cn > 0.999).all(), "nomic cache pipeline mismatch"
del nNet
torch.cuda.empty_cache()
T["probe"] = round(time.time() - t0, 1)
print(f"provenance OK in {T['probe']}s")


groups=290 train=4000 test=1000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5 probe cosine: [1. 1. 1. 1. 1.]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


/home/athilla/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


nomic probe cosine: [1.       0.999999 1.       1.       0.999999]
provenance OK in 39.4s


## Syllabus embeddings + nested experiment

Silabus di-encode protokol retrieval asimetris (passage/search_document) lalu eksperimen
nested dijalankan modul: anchor dibangun ulang per outer fold, threshold di split dalam.


In [3]:
t0 = time.time()
e5t = SentenceTransformer(E5_ID)
ES = np.asarray(e5t.encode(["passage: " + t for t in data.mod_texts], normalize_embeddings=True))
del e5t
torch.cuda.empty_cache()
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
NS = []
with torch.no_grad():
    for s in data.mod_texts:
        e = nTok("search_document: " + s, padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        NS.append((v / v.norm()).cpu().numpy().ravel())
NS = np.array(NS, dtype=np.float32)
del nNet
torch.cuda.empty_cache()
np.save(OUT / "e5_syll.npy", ES)
np.save(OUT / "nomic_syll.npy", NS)
T["syll"] = round(time.time() - t0, 1)
print(f"syllabi encoded {ES.shape} {NS.shape} in {T['syll']}s")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/athilla/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


syllabi encoded (17, 768) (17, 768) in 61.9s


## Nested outer evaluation (modul, teraudit)

Setiap outer fold: anchor + head + threshold hanya dari outer-train. AuditWritten rows
diverifikasi subset dari outer-train di dalam modul.


In [4]:
t0 = time.time()
engine = vp.Engine(data, data.y.copy(), cfg, OUT / "_engine_cache")
res = vp.run_experiment(data, data.y.copy(), cfg, E5U, ES, NOU, NS, engine=engine)
T["experiment"] = round(time.time() - t0, 1)
print(pd.DataFrame(res["fold_table"]).to_string(index=False))
print(f"pooled FULL {res['pooled_full']:+.5f} CI[{res['ci_full'][0]:+.5f},{res['ci_full'][1]:+.5f}] worst-fold {res['worst_fold_full']:+.5f}")
print(f"pooled CTRL {res['pooled_ctrl']:+.5f} CI[{res['ci_ctrl'][0]:+.5f},{res['ci_ctrl'][1]:+.5f}] worst-fold {res['worst_fold_ctrl']:+.5f}")
print(f"dominant block M_{res['dominant_block'][0]:03d} gain {res['dominant_block'][1]:+.5f}; min block {res['min_block']:+.5f}")
print(f"no-chat slice (n={res['no_chat_n']}) FULL gain {res['no_chat_gain']:+.5f}")
print(f"deployed FULL {res['deployed_full']} users, CTRL {res['deployed_ctrl']} users; thrF {res['thr_f']}")
print(f"test actions={int((res['test_actions'] >= 0).sum())}/{n_test} thr_test={res['thr_test']}")
print("ADOPT:", res["gate"])


 fold  thr_c  thr_f  n_c  n_f  oof_anchor  oof_ctrl  oof_full
    0  0.000  0.000   23   48    0.664837  0.665633  0.666235
    1  0.002  0.002    0   13    0.671815  0.671815  0.674109
    2  0.001  0.001    3   49    0.646826  0.646862  0.646921
    3  0.002  0.001    0   21    0.611220  0.611220  0.611353
    4  0.002  0.002   12   38    0.635421  0.634939  0.636061
pooled FULL +0.00071 CI[+0.00017,+0.00132] worst-fold +0.00009
pooled CTRL +0.00009 CI[-0.00016,+0.00036] worst-fold -0.00048
dominant block M_002 gain +0.00150; min block -0.00158
no-chat slice (n=117) FULL gain +0.00000
deployed FULL 169 users, CTRL 38 users; thrF [0.0, 0.002, 0.001, 0.001, 0.002]
test actions=57/1000 thr_test=0.001
ADOPT: False


## Files: kandidat lolos-gate atau byte-copy champion

Deploy test memakai mask non-proteksi V25 + verifikasi geometri di penulis submission.


In [5]:
t0 = time.time()
test_ids = data.user_ids[n_train:]
allowed_test = ~np.isin(test_ids, vp.V25_PROTECTED_USERS)
print(f"test eligible (non-protected): {int(allowed_test.sum())}/{n_test}")
adopt = bool(res["gate"])
if adopt:
    vp.write_submission(CH, OUT / "candidate_v127_primary.csv", res["fin_te"], True, allowed_test)
    print("wrote candidate_v127_primary.csv")
else:
    vp.write_submission(CH, OUT / "champion_unchanged_v127.csv", res["fin_te"], False, allowed_test)
    assert sha256((OUT / "champion_unchanged_v127.csv").read_bytes()).hexdigest() == EXP_FILE
    print("gate failed: champion byte-copy written, hash-verified identical")
REPORT = {"seed": SEED, "splits": cfg.outer_splits, "adopt": adopt,
          "thr_f": res["thr_f"], "thr_c": res["thr_c"], "thr_test": res["thr_test"],
          "pooled_full": res["pooled_full"], "ci_full": res["ci_full"], "worst_fold_full": res["worst_fold_full"],
          "pooled_ctrl": res["pooled_ctrl"], "ci_ctrl": res["ci_ctrl"], "worst_fold_ctrl": res["worst_fold_ctrl"],
          "fold_table": res["fold_table"], "dominant_block": res["dominant_block"], "min_block": res["min_block"],
          "no_chat_n": res["no_chat_n"], "no_chat_gain": res["no_chat_gain"],
          "deployed_full": res["deployed_full"], "deployed_ctrl": res["deployed_ctrl"],
          "test_actions": int((res["test_actions"] >= 0).sum()), "smoke": SMOKE, "timings": T,
          "audit_records": len(res["fit_audit"]),
          "cache_sha": {"e5": sha256(E5U.tobytes()).hexdigest()[:16], "nomic": sha256(NOU.tobytes()).hexdigest()[:16]}}
(OUT / "v127_gate_report.json").write_text(json.dumps(REPORT, indent=1))
(OUT / "NOTES.md").write_text(f"# V127 (nested rebuild)\nadopt={adopt} pooled_full={res['pooled_full']:+.5f} CI=[{res['ci_full'][0]:+.5f},{res['ci_full'][1]:+.5f}] worst_fold={res['worst_fold_full']:+.5f} dom_gain={res['dominant_block'][1]:+.5f} test_actions={int((res['test_actions'] >= 0).sum())} smoke={SMOKE} timings={T}\n")
T["files"] = round(time.time() - t0, 1)
print(json.dumps(REPORT, indent=1)[:1200])


test eligible (non-protected): 992/1000
gate failed: champion byte-copy written, hash-verified identical
{
 "seed": 6161,
 "splits": 5,
 "adopt": false,
 "thr_f": [
  0.0,
  0.002,
  0.001,
  0.001,
  0.002
 ],
 "thr_c": [
  0.0,
  0.002,
  0.001,
  0.002,
  0.002
 ],
 "thr_test": 0.001,
 "pooled_full": 0.0007095365291793237,
 "ci_full": [
  0.00016706341004307814,
  0.0013238588527039847
 ],
 "worst_fold_full": 9.467255060040092e-05,
 "pooled_ctrl": 9.110367477253646e-05,
 "ci_ctrl": [
  -0.00015723176454175925,
  0.000363306763739752
 ],
 "worst_fold_ctrl": -0.00048226265807548615,
 "fold_table": [
  {
   "fold": 0,
   "thr_c": 0.0,
   "thr_f": 0.0,
   "n_c": 23,
   "n_f": 48,
   "oof_anchor": 0.6648365998277808,
   "oof_ctrl": 0.6656334309840857,
   "oof_full": 0.6662347427249882
  },
  {
   "fold": 1,
   "thr_c": 0.002,
   "thr_f": 0.002,
   "n_c": 0,
   "n_f": 13,
   "oof_anchor": 0.671815221581036,
   "oof_ctrl": 0.671815221581036,
   "oof_full": 0.6741089181326334
  },
  {
   "f